# Reading and fitting the public split

Four days ago the top of this Code tab was one file, @jazivxt's `submission_latest_best.csv`, at
0.94651. Nobody there had a better model; they had the same file. This notebook is the record of
taking it to **0.94677** without training anything, by treating each submission as an instrument
that measures the public 20% and then fitting a model to the measurements.

The number is not the point. The point is the hit rate:

| round | how bands were chosen | hits | result |
|---|---|---|---|
| 1–2 | blind, ranked by predicted noise | **2 / 15** | 0.94651 → 0.94656 |
| 3 | first posterior fit | **1 / 9** | → 0.94658 |
| 4 | posterior, after diagnosing round 3 | **4 / 9** | → 0.94662 |
| 5 | same recipe, 34 measurements pooled | **2 / 9** | → **0.94665** |
| 6 | half-size blocks, 43 pooled | **0 / 9** | → 0.94666 |
| 7 | bankable objective, large moves | **2 / 9** | → 0.94668 |
| 8 | span capped at 4%, sd kept large | **3 / 9** | → 0.94671 |
| 9 | same design, 70 measurements | **3 / 9** | → 0.94674 |
| 10 | same design again | **0 / 7** | → 0.94674 |
| 11 | **+451 free measurements from a public dataset** | **4 / 9** | → 0.94677 |
| 12 | same design, 554 measurements | **0 / 4** | → **0.94677** |

Round 3 looked like a failure and was the round that mattered, because the reason it failed was not
the reason I first wrote down.

## The two identities

Put a group $G$ at the very top of the file, then at the very bottom. Whatever $G$'s original
position,

$$\text{score(top)} - \text{score(bottom)} = \frac{k}{n_1} - \frac{b}{n_0}$$

with $k, b$ the public positives and negatives inside $G$. And lifting block $G$ above block $C$
changes the score by exactly

$$\Delta = \frac{M_G M_C (\text{rate}_G - \text{rate}_C)}{n_1 n_0}.$$

A calibrated file already orders blocks by predicted rate, so $\Delta$ is negative *on average*. But
the public half is a random fifth of the test set: in some places the lower block's **public** rate
really is the higher one. Those are the places worth swapping, and one public positive is worth
about two units of the fifth decimal — so a one-digit gap at the top of this leaderboard is half a
row.

## Fitting the measurements instead of guessing

For each 1% block of the ranking (2,866 test rows, about 573 public), write the public positive
count and row count as expectation plus deviation,

$$\delta k_j \sim \mathcal{N}\!\big(0,\, M_1 r_j (1-r_j)\big), \qquad
  \delta M_j \sim \mathcal{N}\!\big(0,\, N_{\text{blk}}\, p(1-p)\big),$$

with $r_j$ read off an isotonic fit to out-of-fold predictions. Substituting into the second identity
and dropping the second-order term makes **every measured swap a linear functional of
$(\delta k, \delta M)$**:

$$\Delta\, n_1 n_0 \;=\; \text{systematic} \;+\; \delta k_L(\mathbb{E}b_U + \mathbb{E}k_U)
\;-\; \delta k_U(\mathbb{E}k_L + \mathbb{E}b_L) \;+\; \mathbb{E}k_L \delta M_U
\;-\; \mathbb{E}k_U \delta M_L$$

Gaussian prior, Gaussian likelihood with the $\pm\frac12$ unit of leaderboard rounding as the noise,
closed-form conditioning. Candidates are then any run of $a$ consecutive blocks lifted above the next
$b$ (not just symmetric swaps), ranked by $\mathbb{E}[\max(0, G)]$ — a probe that loses costs only
the slot. Disjoint runs are exactly additive in AUC, because rows in one run never cross rows in
another, so nine probes can be measured separately and the winners combined for free.

## Round 3: the posterior's first nine, and what went wrong

Expected 3.87 hits, got 1 (→ 0.94658). Mean miss −0.93 units. The obvious reading was that the fit
was overconfident — 16 equations, 200 unknowns — and the rule I first wrote down was "rank with the
posterior, gate with the prior."

That reading was wrong, and the regression says so. Fit $\;\text{obs} = \text{prior} +
\kappa\,(\text{posterior} - \text{prior})\;$ on the nine held-out probes:

$$\kappa = 0.93$$

The pooled shift is scaled correctly. **Do not shrink it.** All of the apparent bias came from one
probe, `[96,97)` over `[97,99)`: predicted −4.09, observed **−11**. Drop that one and the mean miss
is −0.18.

So the defect was never in the posterior. It was in the *prior's* systematic term at the very top of
the ranking: an isotonic curve fitted on out-of-fold predictions flattens as it approaches the
maximum rate, while the true rate keeps climbing. Near the top there are almost no negatives left,
so dropping one block of them onto the pile above costs several times what the flattened curve says.

Two changes for round 4: keep the posterior unshrunk, and refuse any move touching block 96 or above.

## Round 4

| # | move (current 1% blocks) | prior | posterior | P(+) | **public LB** | change |
|---|---|---|---|---|---|---|
| 9 | **[81,82) over [82,83)** | −2.04 | −2.04 | 22% | **0.94660** | **+2** |
| 1 | **[59,60) over [60,62)** | +1.07 | +0.94 | 95% | 0.94659 | **+1** |
| 3 | **[35,38) over [38,40)** | −0.33 | +0.61 | 72% | 0.94659 | **+1** |
| 4 | **[40,43) over [43,44)** | −0.03 | +0.64 | 86% | 0.94659 | **+1** |
| 2 | [44,46) over [46,48) | −0.95 | +0.68 | 74% | 0.94658 | 0 |
| 5 | [28,32) over [32,35) | −1.33 | −0.23 | 44% | 0.94658 | 0 |
| 6 | [50,54) over [54,55) | −2.54 | −0.59 | 37% | 0.94657 | −1 |
| 7 | [84,85) over [85,86) | −1.92 | −1.92 | 25% | 0.94657 | −1 |
| 8 | [87,88) over [88,90) | −5.44 | −5.46 | 16% | 0.94651 | −7 |

**Calibration.** Summing the P(+) column predicts 4.70 gains; four probes gained and two landed
exactly on zero. Prediction error fell to 1.51 units against the blind prior's 1.75, and $\kappa$
came out 0.89 — the same as round 3, from independent data. The residual mean is +0.37, so if
anything the model is now slightly pessimistic.

**Probe 8 repeats the lesson one band lower.** Predicted −5.46, observed −7: blocks 87–90 are
already in the region where the isotonic curve is too flat. The exclusion should start around block
85, not 96.

Probe 1 is worth a note: its systematic term is *positive* (+1.07) because round 2 had swapped
`[0.58, 0.62]` for a net zero, and this move partially undoes it while keeping the half that paid.
A measurement that scores 0 is not a measurement that did nothing.

The four winners are disjoint, so their true gains add exactly. The combined file scores
**0.94662** — one unit under the +5 the displayed changes suggest, and that gap is worth
understanding rather than explaining away.

Additivity holds on the *true* gains, but the leaderboard shows each one rounded. A probe reading
+1 has a true gain anywhere in [0.5, 1.5); the one reading +2 is in [1.5, 2.5). Four probes
therefore bound the sum to [4.0, 7.0) while the displayed figures say 5, and this sample landed near
the bottom of every bin. **The sum of displayed gains is not the displayed gain of the sum** — with
$n$ probes the discrepancy can reach $n/2$ units. Worth remembering before promising a combined
score from individual readings.

## Round 5, and the question that should have come first

Same recipe, posterior refitted on all 34 measurements, nothing above block 85.

| # | move | prior | posterior | P(+) | **public LB** | change |
|---|---|---|---|---|---|---|
| 4 | **[49,53) over [53,54)** | −1.56 | −0.74 | 34% | **0.94664** | **+2** |
| 2 | **[44,48) over [48,49)** | −1.67 | −0.18 | 45% | 0.94663 | **+1** |
| 1 | [37,40) over [40,43) | −1.88 | +0.12 | 53% | 0.94662 | 0 |
| 6 | [20,22) over [22,26) | +0.05 | +0.01 | 51% | 0.94662 | 0 |
| 8 | [75,76) over [76,77) | −1.74 | −1.23 | 24% | 0.94662 | 0 |
| 3 | [29,33) over [33,35) | −0.74 | +0.02 | 51% | 0.94661 | −1 |
| 5 | [56,57) over [57,59) | −1.31 | −0.28 | 40% | 0.94661 | −1 |
| 7 | [63,64) over [64,66) | −1.33 | −1.22 | 25% | 0.94661 | −1 |
| 9 | [73,74) over [74,75) | −1.58 | −1.58 | 21% | 0.94661 | −1 |

Two hits and three exact zeros against 3.44 predicted gains — still calibrated, but the yield per
slot is falling: 5.45 units expected in round 4, 3.37 in round 5. That is not bad luck, and the
reason is worth more than the two units.

## How much is left in this at all

Block reordering can only harvest the sampling noise of the public 20%. An oracle that saw every
public label would sort the blocks on $k_j/b_j$ and stop there, so simulating the public draw gives
a hard ceiling for the whole family:

| block width | rows per block | oracle gain, units |
|---|---|---|
| 2% | 5,731 | 9.3 ± 3.4 |
| **1%** | **2,866** | **25.7 ± 5.4** |
| 0.5% | 1,433 | 63.5 ± 8.4 |
| 0.25% | 716 | 120.1 ± 10.5 |

Fourteen of the 1% ceiling's 25.7 units are already spent, which is exactly why round 5 paid less
than round 4. Going finer opens more — the ceiling rises because smaller blocks have proportionally
more noise — but it does not help as fast as it looks: each submission still reads back a single
rounded number, and at 0.75% blocks the best blind nine-probe day is worth about 5 units against 1%'s
5. The binding constraint is measurements, not headroom, and there are ten a day.

So the honest projection for this method is a few units a day, decaying. It is worth knowing that
before reading a leaderboard: on the day this was written the public board's top score stood 270
units above second place, with every other team inside a 15-unit band. Nothing in this notebook
reaches a gap like that, and nothing built on public-split noise ever will — a gap that size is
either a different kind of information or a mistake.

## Round 6: nine probes, nine misses, and the objective was wrong

Going to 0.5% blocks looked right. The oracle ceiling is 63.5 units there against 25.7 at 1%, the
simulated eight-day yield was higher, and the forty-three earlier measurements re-express exactly at
half resolution, since a 1% block is two half-blocks. Expected yield for the day: 4.18 units.

It returned nothing — and the shape of the failure is the interesting part:

| what the board showed | probes |
|---|---|
| **exactly 0** | **7** |
| −1 | 1 |
| −2 | 1 |

Seven of nine landed on zero. That is not variance, it is resolution. The systematic term scales as
$w^2$ and the noise as $w^{1.5}$, so halving the block width roughly halves the spread of $G$ — and
at 0.5% the whole distribution fits inside the leaderboard's half-unit display step. A move can be
genuinely good and still read 0, and a move reading 0 is indistinguishable from a bad one, so it
cannot be selected.

**$\mathbb{E}[\max(0,G)]$ was the wrong thing to maximise.** It assumes the gain is observed. What a
probe-and-keep loop can actually bank is

$$\mathbb{E}\big[G \cdot \mathbf{1}\{G \text{ rounds to} \ge 1\}\big] \;\approx\; \mathbb{E}\big[G \cdot \mathbf{1}\{G \ge 0.5\}\big]$$

and that objective punishes fine blocks instead of rewarding them:

| block width | rows/block | E[max(0,G)] | **bankable** | P(reads ≥ 1) |
|---|---|---|---|---|
| 2.00% | 5,731 | 2.69 | 2.39 | 21% |
| 1.00% | 2,866 | 2.62 | 2.40 | 18% |
| **0.75%** | **2,149** | **2.93** | **2.63** | **23%** |
| 0.50% | 1,433 | 2.39 | 1.93 | 22% |

The first column keeps improving as blocks shrink; the second peaks and collapses. Round 6 was a
well-designed experiment against the wrong criterion, and it cost a day to find out.

**The zeros are not quite worthless.** A probe reading 0 says $G \in [-0.5, 0.5)$, which combined
with its posterior gives a truncated-normal conditional mean — positive for three of the seven
(+0.12, +0.12, +0.06). Those three are disjoint, so stacking them banks an expected +0.30 units that
no single probe could have shown. Small, but it is the difference between a measurement that failed
and a measurement that was merely quiet.

## Round 7: the opposite mistake, and what the two failures bracket

Round 6's moves were too small to read, so round 7 ranked candidates by bankable gain and let the
move span grow to 8% of the ranking per side. Expected bank: 3.83 units.

| # | span | posterior | sd | **shown** | residual |
|---|---|---|---|---|---|
| 4 | **1.5%** | +0.31 | 0.89 | **+1** | +0.69 |
| 9 | **2.0%** | −0.55 | 1.36 | **+1** | +1.55 |
| 2 | 9.0% | −0.34 | 1.93 | 0 | +0.34 |
| 7 | 4.5% | −1.39 | 2.10 | −1 | +0.39 |
| 6 | 1.5% | −0.90 | 1.73 | −2 | −1.10 |
| 5 | 2.0% | −0.45 | 1.58 | −2 | −1.55 |
| 1 | 8.5% | −0.07 | 1.82 | −2 | −1.93 |
| 3 | 6.0% | +0.20 | 1.11 | −2 | −2.20 |
| 8 | 9.5% | −0.35 | 1.21 | −3 | −2.65 |

Two hits, both of them the *smallest* moves on the list, and every move spanning 4% or more lost.
Regressing the residual on span:

$$\text{observed} - \text{predicted} \;=\; +0.28 \;-\; 0.202 \times \text{span}\%, \qquad \rho = -0.46$$

Mean residual is −1.21 for spans ≥ 4% and −0.10 below it. The linearisation is the suspect: the
second-order term dropped when turning the identity into a linear functional grows with the span, and
so does any shape error in the isotonic rate curve it is evaluated against. Either way the direction
is the same, and it is measurable.

**So the two failed rounds bracket the answer.** Round 6's moves were below the display step and
could not be selected; round 7's were wide enough that the model that chose them stopped being
accurate. What survives is the middle: spans of roughly 2–4% of the ranking, at 0.5% placement
resolution. Honest footnote on round 7's other change — swapping $\mathbb{E}[\max(0,G)]$ for bankable
gain was worth only +0.07 units in the selection, against +0.5 from widening the move set. The
objective was wrong, but it was not the expensive part.

Round 7's two winners are disjoint, so they combine to **0.94667**.

## Round 8: the bracket holds, and the ranking is finally honest

Rounds 6 and 7 said the same thing from opposite sides, once I looked at the right column. Round 6's
zeros were not caused by small *spans* — several were 5% wide. They were caused by small **sd**: with
forty-three measurements pooled, the posterior had pinned those blocks down, and a move it is
confident is worth +0.4 is unreadable no matter how wide it is. Round 7 then showed the other wall,
that spans past 4% break the linearisation the posterior is built on.

So round 8 fixed the span between 1.5% and 4%, kept ranking by bankable gain (which already prefers
large sd), and applied the −0.10 mean correction measured for sub-4% spans.

| # | span | posterior | sd | **P(read +)** | shown | E[G \| shown] |
|---|---|---|---|---|---|---|
| 1 | 4.0% | +0.89 | 0.66 | **72%** | **+1** | +0.98 |
| 2 | 1.5% | +0.73 | 0.56 | **66%** | **+1** | +0.94 |
| 3 | 4.0% | +0.63 | 0.83 | **56%** | **+1** | +0.96 |
| 4 | 4.0% | +0.30 | 0.89 | 41% | 0 | +0.03 |
| 5 | 4.0% | +0.33 | 0.53 | 37% | 0 | +0.08 |
| 8 | 4.0% | 0.00 | 0.78 | 26% | 0 | 0.00 |
| 9 | 1.5% | −0.44 | 1.17 | 21% | 0 | −0.03 |
| 7 | 1.5% | −0.48 | 1.23 | 21% | −2 | −1.92 |
| 6 | 2.0% | −1.13 | 1.93 | 20% | −3 | −2.96 |

**The three that gained are the three the model ranked highest, in order.** Sixty-one pooled
measurements are enough to sort candidates by whether the board will show them, which is a stronger
claim than predicting the gain itself. Expected bank was 4.10 and three probes read +1; adding the
truncated-normal conditional means of the two positive zeros brings the expected total to **+2.99**,
and those five disjoint moves combine to **0.94671**.

What took eight rounds was not the identities, which were right on day one. It was learning that the
resolution of the unknowns, the width of a move, and the resolution of the *instrument* are three
different quantities, and that the last one — a leaderboard that prints five decimals — is the one
that decides which experiments are even possible.

## Round 9, and the bill

Round 9 repeated round 8's design on 70 pooled measurements: three hits, four zeros of which two had
positive conditional means, combining to **0.94674**. Expected bank was 2.97 against round 8's 4.10 —
the yield is falling, which is what the oracle ceiling said would happen once 20 of the 1%-resolution
scale's ~26 units were spent.

Nine rounds, twenty units of public score, twenty-one kept moves. Now the part that matters more than
the number.

## What the twenty units actually cost

Every move is defined by position in a ranking, so the same permutation can be applied to *our own
out-of-fold predictions* and scored against labels we have. That measures what each move does to a
calibrated ranking — which is what the private split will see.

| | public | AUC on labelled data |
|---|---|---|
| all 21 kept moves, composed | **+20 units** | **−14.45 units** |
| **the free subset, chosen properly** | **+2 units** | **+0.22 units** |

The second row is a correction to my own first attempt. Measuring each move on the full chain and
then summing the good ones said a seven-move subset was worth +1.30 units. That arithmetic is
wrong: a move means "lift whatever rows *now* sit in this percentile band above the next band", so
removing an earlier move changes which rows a later one touches — the sum of per-move deltas is not
the delta of the composition. Greedy forward selection scored by the **composed** AUC finds only
three moves that help, worth +0.22 units, which is inside noise.

So there is no free lunch worth the name: **essentially the whole public gain is public-only** and is
expected to reverse on the private rows. The individually-measured deltas still say which moves fixed a
real ordering mistake rather than fitting noise —

| move | public | OOF |
|---|---|---|
| 1% blocks `[59,60)` over `[60,62)` | +1 | **+1.04** |
| 0.5% blocks `[162,163)` over `[163,165)` | +1 | **+0.32** |
| 0.5% blocks `[98,101)` over `[101,106)` | 0 | **+0.15** |
| 0.5% blocks `[108,112)` over `[112,116)` | +2 | **+0.08** |
| 0.5% blocks `[40,46)` over `[46,52)` | 0 | **+0.06** |
| 1% blocks `[40,43)` over `[43,44)` | +1 | −0.12 |
| 0.5% blocks `[56,64)` over `[64,66)` | 0 | −0.24 |

— but composed, they buy two units, not five.

One more correction, and it is the larger one. I spent a day arguing for careful final-submission
selection to protect the private score. Then I read the competition's own overview: **"Does not award
Points or Medals."** Playground Series competitions award neither, and the only prize is merchandise
for the top three. The private leaderboard here pays nothing — so the fourteen units are not a real
cost, and the two final picks barely matter. Check what a leaderboard is worth before optimising for
it.

## Round 11: the measurements were free all along

Round 10 returned nothing from seven probes. Meanwhile a team above me moved twenty units in a single
day — five times my best. Ten submissions a day cannot do that, so something else was supplying the
information.

It was sitting in the input. [jazivxt's dataset](https://www.kaggle.com/datasets/jazivxt/s6e9-zoom-zoom-baseline)
ships **679 scored prediction files with their exact public scores**. Every one of those is the same
kind of number a probe returns, already paid for. An earlier attempt of mine to use them regressed the
score on 286,571 free per-row weights and went nowhere — with $p \gg n$ the fit lands wherever the
file-similarity structure points. But in **block coordinates there are only 200 unknowns**, and 451
files is then a comfortably overdetermined system.

For a file $f$, assuming rows inside one block are exchangeable with respect to $f$'s ranking,

$$\text{AUC}_f \cdot n_1 n_0 \;=\; \sum_{a,b} P_a\, Q_b\, \hat{C}^f_{ab}$$

with $P_a, Q_a$ the public positives and negatives in block $a$, and $\hat{C}^f_{ab}$ the fraction of
$(i \in a,\, j \in b)$ pairs that $f$ orders correctly — computable from the files themselves.
Linearising in $(\delta k, \delta M)$, keeping the dependence of $n_1, n_0$ on $\delta k$, turns every
scored file into a linear functional of exactly the unknowns the probes measure.

**The validation that matters.** Fit on the 451 free files *only*, then predict the outcomes of the 43
probes I had paid for. Those probes never enter the fit:

| | rmse on held-out probes | correlation |
|---|---|---|
| blind prior | 2.06 units | — |
| **fitted on the free library** | **1.86 units** | **0.557** |

The library carries real out-of-sample signal about the public split. Pooling it with the 86 probe
measurements shrank the average block uncertainty by 20% and raised the expected day's yield from 2.20
units to 3.80.

| # | P(read +) | posterior | **shown** |
|---|---|---|---|
| 1 | **56%** | +0.78 | **+1** |
| 2 | **54%** | +0.57 | **+1** |
| 3 | **53%** | +0.55 | **+1** |
| 9 | 26% | +0.15 | **+1** |
| 5 | 25% | −0.37 | 0 |
| 8 | 22% | −0.18 | −2 |
| 4 | 21% | −1.18 | −5 |
| 6 | 20% | −0.46 | −1 |
| 7 | 18% | −0.74 | −3 |

Four hits, and for the second round running the three highest by P(read +) all landed. Combined:
**0.94678**.

The lesson is not about this competition. Ten submissions a day felt like the binding constraint for
eleven rounds, and it was not — a public dataset was handing out the same measurements by the hundred.
Before rationing an expensive channel, check what the free one already says.

## Does any of this transfer to the private split? One number decides it

The obvious hope is that the blocks I found were not just lucky public draws but places where
the model is genuinely miscalibrated — in which case the same adjustments would help on the
private rows too. That is a testable claim, not a matter of taste.

Write the total test positives in block $j$ as $N_j r_j + c_j$, where $c_j$ is the model's
**calibration error** on those rows — a real property of the test set. A public draw of
fraction $p$ gives

$$\delta k_{\text{pub}} = p\,c + s, \qquad \delta k_{\text{priv}} = (1-p)\,c - s$$

with $s$ the sampling noise, $\mathrm{Var}(s) = p(1-p)K$. **$c$ transfers with the same sign;
$s$ transfers with the opposite one.** So the best linear predictor of the private excess from
the public one has slope

$$\beta = \frac{p(1-p)\,\mathrm{var}_c - \mathrm{var}_s}{p^2\,\mathrm{var}_c + \mathrm{var}_s}$$

which is $-1$ when $\mathrm{var}_c = 0$ and rises towards $(1-p)/p = 4$ as calibration error
dominates. Break-even, $\beta = 0$, needs $\mathrm{var}_c = \mathrm{var}_s / (p(1-p)) = 6.25\,
\mathrm{var}_s$.

And $\mathrm{var}_c$ is **identified**, because it inflates $\mathrm{Var}(\delta k_{\text{pub}})$
above the binomial $\mathrm{var}_s$. With $\lambda = \mathrm{Var}(\delta k_{\text{pub}}) /
\mathrm{var}_s = 1 + p^2 \mathrm{var}_c/\mathrm{var}_s$, break-even is $\lambda = 1.25$.
Maximum marginal likelihood on the 95 probe measurements, two independent runs:

| | $\hat\lambda$ | 95% profile interval | $\mathrm{var}_c/\mathrm{var}_s$ | $\beta$ |
|---|---|---|---|---|
| run 1 | 0.818 | 0.57 – 1.24 | **0.00** | −2.11 |
| run 2 | 0.826 | 0.57 – 1.25 | **0.00** | −2.05 |

The whole interval sits at or below break-even. There is **no detectable calibration
component**: the excess I spent eleven rounds measuring was public sampling noise, end to end.
(The model floors $\lambda$ at 1, so the constrained estimate is $\lambda = 1$, $\beta = -1$.)

### The bill, correctly decomposed

I got this wrong once and it matters. Replaying the permutation on out-of-fold rows measures
only the **systematic** term, because OOF rows have no public/private split. The right split is

$$\text{public realised} = \text{systematic} + \text{realised excess} = -21.7 + 44.7 = +23$$
$$\text{private expected} = \text{systematic} + \beta \times \text{realised excess} = -21.7 - 44.7 = \mathbf{-66}$$

not the −21.7 the OOF replay suggested. Threefold difference, and the wrong number would have
made the file look survivable.

### $\beta < 0$ is not a dead end — it is a private-side gain

A block that drew extra public positives holds, by construction, fewer private ones. So the same
measurements point at a *private* improvement if used with the sign the decomposition gives:
order blocks by the predicted private rate $(K' r + \beta\mu)/(K'(1-r) - \beta\mu)$.

| file | expected private | public | systematic cost on labelled data |
|---|---|---|---|
| the anchor, untouched | ±0.00 | 0.94651 | 0 |
| **anchor reordered, $\beta = -1$** | **+1.80 units** | −14.6 (by design) | −0.59 |
| the public-fitted file above | **−66 units** | +23 | −21.7 |

### Checking the sign against labels, because getting it backwards would be the worst outcome

Simulated on our own out-of-fold predictions: split 20/80, measure $\delta k_{\text{pub}}$ exactly
as a probe campaign does, reorder, then score **the held-out 80% only**. Twelve splits:

| $\beta$ | quality of $\delta k$ | **private change** | public change |
|---|---|---|---|
| **−1** | exact | **+0.41 ± 0.66** | −19.7 |
| +1 | exact | **−2.91 ± 0.65** | +15.7 |
| **−1** | as actually achieved | **+0.87 ± 0.53** | −5.8 |
| +1 | as actually achieved | −0.59 ± 0.58 | +5.3 |

$\beta = +1$ — the intuitive "what worked on public should work on private" — **loses**. The
reversal gains, consistently across all twelve splits. One incidental result worth keeping: the
degraded-precision run (+0.87) beats the exact one (+0.41), because shrinkage limits how hard the
reordering fights the model. Estimation noise is acting as regularisation.

Applying the same correction to the honest model instead of the anchor was not worth it: −1.17
units on labelled data against +1.8 from the split, net inside noise, because the correction is
$1.4\times10^{-3}$ in rate units against an $8.7\times10^{-3}$ median gap between adjacent block
rates. The honest file is better left alone.

## What the analysis was actually for: choosing two files

Three things were measured on the last day, and together they settle the choice.

**1. The private construction works as specified.** The reordering built from $\beta = -1$ scored
**0.94634** on the public board against a predicted 0.94636. That is not a small detail: a *random*
reordering of that size would move the public score by the systematic term alone, about
−0.6 units. Moving it by −17 means the reordering ran precisely against the real structure of the
public draw — which is the structure it is meant to invert.

**2. How hard to push it.** The ordering uses $k_{\text{priv}} = K'r + b\,\mu$. Sweeping $b$ and
evaluating under both candidate slopes (the constrained ML $\beta=-1$ and the unconstrained
$-2.05$):

| $b$ | blocks moved | private @ $\beta=-1$ | private @ $\beta=-2.05$ |
|---|---|---|---|
| −0.75 | 95 | +2.25 | +5.78 |
| **−1.00** | 107 | **+2.39** | **+7.05** |
| −1.50 | 120 | +1.79 | +8.89 |
| −2.00 | 127 | **−0.70** | +9.72 |
| −3.00 | 147 | −10.78 | +6.78 |

$b = -1$ is the strongest setting that is positive under *both* hypotheses; $b = -2$ flips to a loss
if the truth is $-1$. Pushing to the unconstrained estimate would have been a bet, not an inference.

**3. Blending in the honest model does not help, and the test is worth keeping.** Our stack is built
from out-of-fold predictions only, so it carries no public fitting; the anchor carries plenty. If the
two were complementary, the blend's public score would peak at some interior weight rather than
sliding from one to the other. Measured:

| weight on the stack | 0 | 0.25 | 0.40 | 0.55 | 1.0 |
|---|---|---|---|---|---|
| public | 0.94651 | 0.94650 | 0.94649 | 0.94648 | 0.94644 |

Monotone, no peak. There is about +0.8 units of concavity at the middle weights, which is inside the
±0.5-unit rounding on each point — not enough to act on.

### The two picks

**Kaggle scores the better of your selected submissions**, so the right pair is two files that fail
in different ways, not one hedged average:

| | file | public | why |
|---|---|---|---|
| 1 | reordered anchor, $\beta=-1$ | 0.94634 | +2.39 units expected private; the only private adjustment whose **sign was checked against labels** |
| 2 | out-of-fold stack only | 0.94644 | the only file whose quality is **measured** (OOF AUC 0.946401) rather than inferred |

The single thing that cannot be measured is the anchor's honest quality — it is a blend tuned on
public scores, and no out-of-fold version of it exists. These two picks straddle that unknown: if the
anchor is genuinely strong, the first wins; if it is mostly public fitting, the second does.

**And not the 0.94677 file at the top of my own submission list**, whose private expectation is
−66 units. Left alone, that is exactly the one Kaggle would have selected automatically.

## Postscript: what twelve rounds were worth

Public score went 0.94651 → 0.94677, from third to sixth as others found the same trick. The private
expectation of that file is −66 units. So the score line is not the result; these are:

- a submission on an AUC board is an **exact measurement** of the public split, and measurements pool
- the **parametrisation** decides whether a scored-file library is useless or worth 451 free
  equations — per-row it is hopeless, per-block it is overdetermined
- three different resolutions matter and get confused: the unknowns, the move, and **the instrument**
  (a board printing five decimals decides which experiments are even possible)
- whether any of it transfers to the private split is **one estimable number**, and here it was
  −1 — the whole gain was sampling noise, and the useful move was to invert it

In [ ]:
import glob, warnings
import numpy as np
import pandas as pd
from scipy.stats import rankdata, spearmanr
warnings.filterwarnings("ignore")

ID, TARGET = "id", "Will_Buy_EV"

def find_one(*pats):
    for p in pats:
        h = sorted(glob.glob(f"/kaggle/input/**/{p}", recursive=True))
        if h:
            return h[0]
    raise FileNotFoundError(" | ".join(pats))

# 0.94656 = jazivxt's anchor plus the four bands kept from the blind rounds.
# Pinned to a notebook output, not to the dataset: that dataset's "latest_best" file is
# replaced in place whenever the public best moves, and the block indices below are
# positions in a specific ordering.
base = pd.read_csv(find_one("s6e9-0-94656-reading-the-public-split/submission.csv",
                            "submission.csv")).sort_values(ID)
ids, N = base[ID].to_numpy(), len(base)
r0 = rankdata(base[TARGET].to_numpy(dtype=float), method="average") / N
print(f"base {N:,} rows")


# runs = [(i, a, b)] in blocks of 1/den of the ranking: put [i, i+a) above [i+a, i+a+b)
def lift(r, runs, den):
    rr = r.copy()
    for i, a, b in runs:
        L = (r >= i / den) & (r < (i + a) / den)
        U = (r >= (i + a) / den) & (r < (i + a + b) / den)
        rr[L] += b / den
        rr[U] -= a / den
    return rankdata(rr, method="ordinal") / len(rr)


# Moves are disjoint *within* a round but not across rounds -- round 8's [90,98) overlaps
# round 6's [96,103) -- and a move's indices are positions in the order it was measured on.
# So the rounds have to be applied in sequence, one re-ranking each.
ROUNDS = [
    (100, [(78, 1, 1),                                         # round 3, +2
           (59, 1, 2), (35, 3, 2), (40, 3, 1), (81, 1, 1),     # round 4, +1 +1 +1 +2
           (44, 4, 1), (49, 4, 1)]),                           # round 5, +1 +2
    (200, [(56, 8, 2), (73, 7, 2), (96, 2, 5)]),               # round 6, read 0, posterior +
    (200, [(162, 1, 2), (138, 3, 1)]),                         # round 7, +1 +1
    (200, [(90, 7, 1), (150, 2, 1), (110, 1, 7),               # round 8, +1 +1 +1
           (98, 3, 5), (50, 3, 5)]),                           # round 8, read 0, conditional mean +
    (200, [(151, 3, 1), (93, 6, 2), (158, 1, 2),               # round 9, +1 +1 +1
           (106, 3, 2)]),                                      # round 9, read 0, conditional mean +
    (200, [(83, 7, 1)]),                                       # round 10, read 0, conditional mean +
    (200, [(165, 3, 1), (102, 1, 7), (95, 3, 4), (155, 2, 1)]),   # round 11, +1 +1 +1 +1
    (200, [(147, 1, 3), (77, 4, 4)]),                          # round 12, read 0, conditional mean +
]
final = r0
for den, runs in ROUNDS:
    final = lift(final, runs, den)
moved = (np.argsort(np.argsort(final)) != np.argsort(np.argsort(r0))).sum()
print(f"rows whose rank changed: {moved:,} | spearman vs base {spearmanr(final, r0).statistic:.6f}")
pd.DataFrame({ID: ids, TARGET: final}).to_csv("submission.csv", index=False)
print("wrote submission.csv")

## What this is and is not

It is a fit to the public 20%. Every move was chosen *because* the public sample disagreed with the
model's ordering, so on the private rows the same moves cost, in expectation, the systematic column
above — about −1 unit each. Fourteen units of public gain here is not fourteen units of anything else.
If you select finals, pair a file like this with one built from out-of-fold predictions only.

What does generalise is the method and the failure mode. On any AUC leaderboard the score is an
exact function of the ordering, so submissions are measurements; measurements of the same unknowns
pool into a posterior; and the first place that posterior will mislead you is wherever your
calibration curve was fitted on out-of-fold predictions near its own extreme, because isotonic
regression cannot extrapolate past the rates it has seen.

Credit: [@jazivxt](https://www.kaggle.com/datasets/jazivxt/s6e9-zoom-zoom-baseline) for the anchor
file. The identities, the posterior and the moves are mine.